In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import math

In [ ]:
import matplotlib.colors as mcolors

In [ ]:
sc.settings.verbosity = 3 
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, transparent = True, format = 'pdf', vector_friendly = True)

In [ ]:
figure = "Figure_5"

In [ ]:
sc.settings.figdir = './Figure_plots/'+figure

In [ ]:
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:blueberry'], as_cmap = True)

# input files

In [ ]:
adata = sc.read_h5ad('./h5ad/analysis_250528_f/Smed_L78-L47_20250523_Annotated.h5ad')

In [ ]:
adata

In [ ]:
# load the results from orthofinder (all genes identified by orthofinder, expressed in all cell types and not only the neoblasts)
# provided as data S8
cc_df = pd.read_csv('Peron_et_al_2025_Fig5_20260117_cellcycle_genes_v1_based_on_orthofinder_AND_expressed_in_neoblasts.txt', header = None)

In [ ]:
cc_df

In [ ]:
cc_li = list(cc_df[0])
cc_li

In [ ]:
sc.tl.score_genes(adata, gene_list=cc_li, score_name='score_cc')

In [ ]:
sc.pl.umap(adata, color = 'score_cc', cmap = 'Purples', s=5, show = True )
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.5)) 

sc.pl.violin(
    adata,
    keys='score_cc',
    groupby='annotated_names',
    stripplot=False,
    ax=ax,             
    show=False          
)

plt.setp(ax.get_xticklabels(), rotation=90, ha='right')  
plt.tight_layout()
plt.savefig('./Figure_plots/'+ figure + '/' + figure + "_cc_score_violin.pdf")
plt.show()


In [ ]:
# cell cycle genes represented in the scheme of the figure
cc = {'APC complex': ["h1SMcG0001410", "h1SMcG0003111", "h1SMcG0004663", "h1SMcG0005225", "h1SMcG0005227"
                      , "h1SMcG0005232", "h1SMnG0008807", "h1SMcG0014835", "h1SMcG0014907", "h1SMcG0016930"
                      , "h1SMnG0024113", "h1SMnG0027366"]
      ,'Cdh1': ['h1SMcG0005288'], 'CDC20-l': ['h1SMcG0009584']
      ,'Aurora kinase': ["h1SMcG0002748", "h1SMcG0003809", "h1SMcG0003826", "h1SMcG0004538", "h1SMcG0006068"]
      , 'Cyclin': ["h1SMcG0002811", "h1SMcG0004348", "h1SMcG0010023", "h1SMcG0016400"]
      ,'CDK': ["h1SMcG0005366", "h1SMcG0005454", "h1SMcG0007819", "h1SMcG0008190", "h1SMcG0015408"
               , "h1SMcG0015412", "h1SMcG0018655", "h1SMcG0009776", "h1SMcG0017639", "h1SMcG0022320", "h1SMnG0024124"]
      ,'E2F': ["h1SMcG0010707", "h1SMnG0017612"]
      ,'PP1': ["h1SMcG0000623", "h1SMcG0002168", "h1SMcG0002537", "h1SMcG0002891", "h1SMcG0002892", "h1SMcG0004160"
               , "h1SMcG0005583", "h1SMcG0011997", "h1SMcG0012256", "h1SMcG0016362", "h1SMcG0021223", "h1SMnG0003510"]
      ,'PP2A-2': ["h1SMcG0000551", "h1SMcG0003463", "h1SMcG0010177", "h1SMcG0014590", "h1SMcG0015068", "h1SMcG0016867"
                  , "h1SMcG0020570"]
     }

In [ ]:
# check the expression of some cell cycle genes

# adata only contains the 20000 highly variable genes, so many are missing
# adata.raw contains the normalised counts for all genes
adata_from_raw = adata.raw.to_adata() 

for pathway, genes in cc.items():
    for gene in genes:
        if gene in adata.raw.var.index: 
            sc.pl.umap(
                adata_from_raw,
                color=gene,
                cmap=umap_cmap,
                s=10,
                title=f"{pathway} - {gene}"
        )

In [ ]:
# list of selected cell cycle genes
li = [gene for genes in cc.values() for gene in genes]
len(li)

In [ ]:
# cc_li: cell cycle genes expressed in neoblasts
li_neo = [i for i in li if i in cc_li]
len(li_neo)

In [ ]:
li_neo

In [ ]:
# keep only genes expressed in neoblasts
cc_filtered = {pathway: [gene for gene in genes if gene in li_neo] 
               for pathway, genes in cc.items()}
cc_filtered = {k: v for k, v in cc_filtered.items() if v} # remove empty lists
print(cc_filtered)

In [ ]:
# check the expression 

# adata only contains the 20000 highly variable genes, so many are missing
# adata.raw contains the normalised counts for all genes
adata_from_raw = adata.raw.to_adata() 

for pathway, genes in cc_filtered.items():
    for gene in genes:
        if gene in adata.raw.var.index: 
            sc.pl.umap(
                adata_from_raw,
                color=gene,
                cmap=umap_cmap,
                s=10,
                title=f"{pathway} - {gene}"
        )

In [ ]:
# dict containing only the stronger expression
list_plot = {'APC-C': ['h1SMcG0003111', 'h1SMcG0004663'], 
             'Cdh1': ['h1SMcG0005288'], 'CDC20-l': ['h1SMcG0009584'], 
             'Aurora-K': ['h1SMcG0002748'], 
             'Cyclin': ['h1SMcG0002811'], 
             'CDK': [ 'h1SMcG0008190', 'h1SMcG0015412'], 
             'E2F': ['h1SMcG0010707'], 'PP2A-2': ['h1SMcG0003463']}
li_filt_genes = [gene for genes in list_plot.values() for gene in genes]

In [ ]:
li_filt = [(pathway, gene) for pathway, genes in list_plot.items() for gene in genes]

n = len(li_filt)
ncols = 5
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(4*ncols, 4*nrows),
    dpi=300
)

axes = axes.flatten()
adata_from_raw = adata.raw.to_adata()

for i, (pathway, gene) in enumerate(li_filt):
    sc.pl.umap(
        adata_from_raw,
        color=gene,
        cmap=umap_cmap,
        s=10,
        ax=axes[i],                   
        show=False,                   
        title=f"{pathway}: {gene}"  
    )
    axes[i].set_axis_off() 
    axes[i].title.set_fontsize(18)
    axes[i].title.set_fontweight('bold')


# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(f'./Figure_plots/{figure}/{figure}_umaps.pdf')
plt.show()



In [ ]:
li_filt_genes

In [ ]:
adata.raw.var.index

In [ ]:
# plot of the other genes
cc_filtered_2 = {pathway: [gene for gene in genes if gene not in li_filt_genes and gene in adata.raw.var.index] 
               for pathway, genes in cc.items()}
cc_filtered_2 = {k: v for k, v in cc_filtered_2.items() if v} # remove empty lists
print(cc_filtered_2)

In [ ]:
li_filt = [(pathway, gene) for pathway, genes in cc_filtered_2.items() for gene in genes]

n = len(li_filt)
ncols = 6
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(4*ncols, 4*nrows),
    dpi=100
)

axes = axes.flatten()
adata_from_raw = adata.raw.to_adata()

for i, (pathway, gene) in enumerate(li_filt):
    sc.pl.umap(
        adata_from_raw,
        color=gene,
        cmap=umap_cmap,
        s=10,
        ax=axes[i],                   
        show=False,                   
        title=f"{pathway}: {gene}"  
    )
    axes[i].set_axis_off() 
    axes[i].title.set_fontsize(16)
    axes[i].title.set_fontweight('bold')


# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(f'./Figure_plots/{figure}/{figure}_umaps_cc_rest.pdf')
plt.show()



In [ ]:
cc_li

## Correlation between cell cycle score and neoblast score

In [ ]:
# overlap with the neoblast score
clusteringlayer = 'leiden_2.5'
markers_w = pd.DataFrame(adata.uns['rank_genes_groups_wilcox_'+clusteringlayer]['names']).head(50)
li_neo_v2 = list(set(markers_w['0'].head(50).to_list() + markers_w['1'].head(50).to_list()))

li_neo_cc = [i for i in li_neo_v2 if i in cc_li]
len(li_neo_cc)

In [ ]:
def plot_obs_scatter(adata, x_col, y_col, method="pearson", hue_col=None, palette="viridis", save_path=None, fontsize = 8):
    df = adata.obs.copy()
    corr = df[x_col].corr(df[y_col], method=method)
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    ax.grid(False)
    if hue_col and hue_col in df.columns:
        sc = sns.scatterplot(
            data=df, x=x_col, y=y_col,
            hue=hue_col, palette=palette,
            s=2, edgecolor=None, zorder=3, legend=False, 
            rasterized=True
        )
        sns.regplot(
            data=df, x=x_col, y=y_col,
            scatter=False,
            line_kws={"color": "black", "zorder": 4}
        )
        if pd.api.types.is_numeric_dtype(df[hue_col]):
            norm = plt.Normalize(vmin=df[hue_col].min(), vmax=df[hue_col].max())
            sm = plt.cm.ScalarMappable(cmap=palette, norm=norm)
            sm.set_array([])
            cbar = plt.colorbar(sm, ax=ax)
            cbar.set_label(hue_col, fontsize=fontsize)
            cbar.ax.tick_params(labelsize=fontsize)
    else:
        sns.regplot(
            data=df, x=x_col, y=y_col,
            scatter_kws={"s": 20, "alpha": 0.7, "zorder": 3},
            line_kws={"color": "black", "zorder": 4}
        )
    ax.tick_params(axis='both', labelsize=fontsize) 
    ax.set_xlabel(x_col, fontsize=fontsize)        
    ax.set_ylabel(y_col, fontsize=fontsize)         
    ax.text(
        0.05, 0.95, f"{method}: {corr:.3f}",
        transform=ax.transAxes, ha="left", va="top", 
        fontsize=fontsize + 2 
    )
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    
    plt.show()

In [ ]:
# define the colormap
colors = ['#f08080', '#4a3a4a', '#6a8faf', '#b0c4de', "#b0e0e6"]
stops  = [0, 25, 50, 255]
positions = [s / 255 for s in stops]

custom_cmap = mcolors.LinearSegmentedColormap.from_list(
    name="custom_cmap",
    colors=list(zip(positions, colors)),
    N=256)


In [ ]:
# check color map
gradient = np.linspace(0, 1, 256)
gradient = np.vstack((gradient, gradient))

plt.figure(figsize=(6, 1))
plt.title("Custom Purples Colormap")
plt.imshow(gradient, aspect='auto', cmap=custom_cmap)
plt.axis('off')
plt.show()

In [ ]:
plot_obs_scatter(adata, 'neoblast_score', 'score_cc', 
                 hue_col='n_counts', method="spearman", palette=custom_cmap, fontsize = 12,
                 save_path= './Figure_plots/'+figure + "/correlation_neo_cc.pdf" )